# Baseline_v20_v1 — 五折推理

使用 v20_v1 训练产生的 `fold_0/best.pt` 至 `fold_4/best.pt`，不再使用旧的单份 v14 权重。4349 个弱标签 Study 分五折，58 个金标始终不训练，并用于每轮验证和最佳 epoch 选择。五折 sigmoid 概率等权平均；各模型数值步骤仍与 v14 一致：float32 权重/输入、fp16 autocast、全部候选窗口、原始裁剪和分位数、encoder chunk size=16。融合后的输出会与旧单模型不同。

挂载同一目录下的 fold0.pt 至 fold4.pt（或五份 best.pt）、DINOv2 Small 和比赛数据。自动发现必须恰好有五份权重，否则明确填写 `CHECKPOINT_PATHS`。checkpoint 必须覆盖 fold 0..4，并具有相同的划分/标签指纹、预处理和 metadata 配置；每折单独使用自己训练集拟合的 coverage 阈值。

每张 GPU 独立持有五模型，处理独立 Study 分片。每个 Study 的 DICOM 原始字节只读一次，pydicom 解析/解码沿用 v14；16 个 I/O 线程、最多 2 个 Study 预取。仅当五个 slot 选择的 Series UID 相同时复用预处理，避免把某折的序列选择错误用于其他折。没有 uint8 量化、分位数近似、half 权重或自定义 DICOM 解码。

本地 CPU 验证不能替代 Kaggle T4 的完整运行；五模型会增加计算时间，需实际确认显存、9 小时限制及分数。训练和 OOF/金标评估见本目录 README。


In [ ]:
"""RSNA Knee 数据加载与 140 mm physical crop 预处理模块。"""

import hashlib
import json
import os
import uuid
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from pydicom.errors import InvalidDicomError
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


DEFAULT_ROOT = Path("/root/RSNA/rsna-knee-abnormality-detection")
DEFAULT_IMAGE_SIZE = 336
DEFAULT_CROP_MM = 140.0
# 一个 Study 预测 12 个标签；五个 slot 按方位/Fluid Sensitive 偏好选不同序列。
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = ["Sagittal_fluid", "Sagittal_second", "Coronal_fluid", "Coronal_second", "Axial"]
SLOT_SPECS = [("Sagittal", 1, 23), ("Sagittal", 0, 17), ("Coronal", 1, 15), ("Coronal", 0, 10), ("Axial", None, 15)]
CANDIDATE_BUDGETS = [spec[2] for spec in SLOT_SPECS]
CACHE_SCHEMA = 1
SERIES_SELECTION_VERSION = "header_quality_v1"
QUALITY_COLUMNS = ["quality_usable", "quality_header_errors", "quality_geometry_errors",
                   "quality_unique_slices", "quality_coverage_mm", "quality_pixel_spacing_mm",
                   "quality_slice_spacing_mm", "quality_fov_mm"]
# 每个序列对应 11 个数值元数据，顺序必须与 load_series 构造 features 的顺序一致。
SERIES_FEATURES = ["fluid_sensitive", "fat_suppression", "t1", "pd", "t2", "contrast_unknown", "te", "tr", "pixel_spacing", "slice_spacing", "coverage"] #翻译：
#MRI 序列特征：      液体敏感；         脂肪抑制； T1 加权；质子密度加权；T2 加权；对比或加权类型未知；回波时间；重复时间；像素间距；相邻切片间距；扫描覆盖范围。

def _count_dicoms(path):
    return sum(1 for _ in Path(path).glob("*.dcm"))


def _prepare_series_df(series_df, image_root):
    # 在原始序列表上补路径、槽位和切片数，方便后续按 Study/槽位选序列。
    series_df = series_df.copy()
    series_df["SeriesPath"] = [Path(image_root) / str(study) / str(series) for study, series in zip(series_df.StudyInstanceUID, series_df.SeriesInstanceUID)]
    series_df["NumSlices"] = [_count_dicoms(path) for path in series_df.SeriesPath]
    return series_df


def load_metadata(root=DEFAULT_ROOT):
    # Study 标签表与序列描述表分别读取；实际 DICOM 只在 __getitem__ 时按需解码。
    root = Path(root)
    train_df, test_df = pd.read_csv(root / "train.csv"), pd.read_csv(root / "test.csv")
    train_series_df = _prepare_series_df(pd.read_csv(root / "train_series.csv"), root / "train_series")
    test_series_df = _prepare_series_df(pd.read_csv(root / "test_series.csv"), root / "test_series")
    return train_df, train_series_df, test_df, test_series_df, pd.read_csv(root / "sample_submission.csv")


def get_sorted_dicom_info(series_path):

    """使用患者坐标排序，并把主切片方向统一为患者坐标轴的正方向。"""

    # 获取当前MRI序列文件夹下所有DICOM文件路径
    files = list(Path(series_path).glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"No DICOM files found in series: {series_path}")
    # 读取所有DICOM文件的头信息，但不读取像素数据，节省内存
    headers = [inference_read_header(file, stop_before_pixels=True) for file in files]
    # 取第一张切片的ImageOrientationPatient，表示该图片在患者空间中的行方向和列方向   #ImagePositionPatient表示这一张图片左上角在患者三维空间的位置
    orientation = np.asarray(headers[0].ImageOrientationPatient, dtype=np.float32)
    # 前3个数表示图片行方向，后3个数表示图片列方向，两个方向叉乘得到切片法向量
    normal = np.cross(orientation[:3], orientation[3:])
    # 将法向量归一化，使其长度变为1，方便后续进行空间投影计算
    normal /= np.linalg.norm(normal)
    # 判断法向量主要方向是否为负，如果是则翻转，保证所有序列方向统一
    if normal[np.argmax(np.abs(normal))] < 0:
        # 反转法向量方向
        normal = -normal
    # 将每张切片的患者空间坐标投影到切片法向量方向，得到每张slice的真实空间位置
    positions = np.asarray([np.dot(np.asarray(header.ImagePositionPatient, dtype=np.float32), normal) for header in headers])
    # 根据真实空间位置从小到大排序，得到正确的slice顺序
    order = np.argsort(positions)
    # 按照排序后的索引，同时返回：
    # 1. 排序后的DICOM文件 2. 对应空间位置 3. 对应header信息
    return [files[i] for i in order], positions[order], [headers[i] for i in order]


def sort_dicom_files(series_path):
    # 只需要已排序文件列表时的便捷包装。
    return get_sorted_dicom_info(series_path)[0]

#读取一张DICOM MRI切片，将其转换成标准的二维numpy图像矩阵，并处理少数异常DICOM文件和灰度方向问题。
def read_dicom(path):
    dicom = inference_read_dataset(path)
    # 尝试直接通过pydicom解析PixelData得到图像矩阵
    try:
        # pixel_array会自动根据DICOM头信息解析像素，并转换成numpy数组
        image = dicom.pixel_array.astype(np.float32)
    # 如果标准解析失败，进入手动恢复流程
    except ValueError as error:
        # 获取图像高度和宽度，后续用于手动reshape像素数据
        rows, columns = int(dicom.Rows), int(dicom.Columns)
        # 获取帧数，多帧DICOM没有该字段时默认认为只有1帧
        frames = int(getattr(dicom, "NumberOfFrames", 1) or 1)
        # 获取每个像素包含几个采样值，例如RGB可能是3，MRI通常是1
        samples = int(getattr(dicom, "SamplesPerPixel", 1) or 1)
        # 计算理论上的像素总数量
        pixel_count = rows * columns * frames * samples
        # 直接读取DICOM中的原始像素字节数据
        raw = dicom.PixelData
        # 获取DICOM传输格式，用于判断PixelData是否经过压缩
        transfer_syntax = getattr(getattr(dicom, "file_meta", None), "TransferSyntaxUID", None)
        # 判断当前DICOM是否为未压缩格式
        uncompressed = transfer_syntax is None or not transfer_syntax.is_compressed

        # 只有满足以下条件才允许手动解析：
        # 1. 未压缩
        # 2. 单帧
        # 3. 单通道
        # 4. 原始字节数量符合预期
        if not uncompressed or frames != 1 or samples != 1 or len(raw) not in (pixel_count, pixel_count + 1):

            # 如果情况复杂，无法安全恢复，则直接抛出异常
            raise ValueError(f"Cannot decode DICOM {path}: {error}") from error

        # 根据PixelRepresentation判断像素是否为有符号整数
        # 0代表unsigned，1代表signed
        dtype = np.int8 if int(getattr(dicom, "PixelRepresentation", 0) or 0) else np.uint8

        # 从原始字节创建numpy数组，并恢复成二维图像
        image = np.frombuffer(raw[:pixel_count], dtype=dtype).reshape(rows, columns).astype(np.float32)

    # 根据DICOM标准应用灰度线性变换
    # 输出值 = 原始像素值 × RescaleSlope + RescaleIntercept
    image = image * float(getattr(dicom, "RescaleSlope", 1)) + float(getattr(dicom, "RescaleIntercept", 0))

    # MONOCHROME1表示像素越大越黑，需要反转成普通医学图像亮度方向
    if getattr(dicom, "PhotometricInterpretation", "") == "MONOCHROME1":
        # 使用最大值+最小值减去原像素，实现灰度反转
        image = image.max() + image.min() - image

    # 返回最终处理好的二维MRI切片
    return image

#读取指定的MRI切片，并且在遇到损坏DICOM时，用距离最近的正常切片替代，最后把所有slice组合成3D volume。注意这里的volume是一个numpy数组，shape为[S,H,W]，其中S是切片数，H和W是每张切片的高度和宽度。不是channle为3
def _read_sampled_slices(files, indices):
    """读取采样切片；单张损坏时用空间上最近的可读切片替代。"""
    # cache 避免重复解码同一张；invalid 避免对已知坏文件反复尝试。
    cache, invalid = {}, set()
    images = []
    for index in indices:
        index = int(index)
        # 按切片索引距离从近到远尝试，直到找到可解码的替代切片。
        candidates = sorted(range(len(files)), key=lambda candidate: abs(candidate - index))
        for candidate in candidates:
            if candidate in cache:
                image = cache[candidate]
                break
            if candidate in invalid:
                continue
            try:
                image = read_dicom(files[candidate])
                cache[candidate] = image
                break
            except (ValueError, OSError, EOFError, InvalidDicomError):
                invalid.add(candidate)
        else:
            raise RuntimeError(f"No decodable DICOM slices in series: {Path(files[0]).parent}")
        images.append(image)
    return np.stack(images)


def _sample_by_position(positions, num_slices):
    """在真实物理覆盖范围内均匀采样，而不是只按切片编号采样。"""
    # targets 是等间隔物理坐标；每个目标位置选最近的真实 DICOM 切片。
    targets = np.linspace(positions[0], positions[-1], num_slices)
    return np.abs(positions[:, None] - targets[None, :]).argmin(axis=0)


#从一个MRI序列的DICOM header中读取像素空间大小（PixelSpacing），用于后续把“毫米尺度的crop”转换成“像素尺度的crop”。
def _get_pixel_spacing(headers, series_path):

    """读取有效的行、列 PixelSpacing；优先使用堆栈中央切片。"""

    # 计算DICOM序列中间位置的slice索引，优先检查中间切片
    middle = len(headers) // 2
    # 按距离中间切片的远近排序，生成检查顺序
    search_order = sorted(range(len(headers)), key=lambda index: abs(index - middle))
    # 按照从中间到两边的顺序遍历每一张slice的header
    for index in search_order:

        # 获取当前slice的PixelSpacing字段，如果不存在则返回None
        value = getattr(headers[index], "PixelSpacing", None)

        # 如果当前slice没有PixelSpacing信息，跳过这一张slice
        if value is None or len(value) < 2:
            continue

        # 将PixelSpacing转换成numpy数组，只保留行方向和列方向的两个spacing值
        spacing = np.asarray(value[:2], dtype=np.float32)

        # 检查spacing是否有效：1. 所有值不是NaN/inf 2. 所有值大于0
        if np.all(np.isfinite(spacing)) and np.all(spacing > 0):
            # 返回有效的像素间距，单位通常是mm/pixel
            return spacing
    # 如果遍历所有slice都没有找到有效PixelSpacing，则说明该序列无法确定物理尺寸
    raise ValueError(f"Series has no valid PixelSpacing: {series_path}")


def physical_center_crop(volume, pixel_spacing, crop_mm=DEFAULT_CROP_MM):

    """按真实毫米尺度中心裁剪；原始 FOV 不足时先对称补零以保持尺度。"""

    # 检查输入volume是否为三维MRI数据，格式应该是 [slice数量, height, width]
    if volume.ndim != 3:
        # 如果不是三维数据，说明输入格式错误，直接报错
        raise ValueError(f"Expected volume [S,H,W], got shape {volume.shape}")
    # 检查裁剪尺寸是否有效，crop_mm必须是正数
    if not np.isfinite(crop_mm) or crop_mm <= 0:
        # 如果crop_mm为负数、0或者无穷/NaN，则无法进行裁剪
        raise ValueError(f"crop_mm must be positive, got {crop_mm}")
    # 将pixel_spacing转换成numpy数组，方便后续数学计算
    spacing = np.asarray(pixel_spacing, dtype=np.float32)

    # 检查PixelSpacing是否合法：必须包含两个值，并且不能为NaN、inf，同时两个方向都必须大于0
    if spacing.shape != (2,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
        # PixelSpacing异常时无法从毫米转换到像素，因此停止处理
        raise ValueError(f"Invalid PixelSpacing: {pixel_spacing}")
    # 根据真实毫米大小计算需要裁剪的像素高度 # 公式：目标像素数量 = crop毫米数 / 每个像素代表的毫米数
    target_height = max(1, int(round(crop_mm / float(spacing[0]))))
    # 根据真实毫米大小计算需要裁剪的像素宽度
    target_width = max(1, int(round(crop_mm / float(spacing[1]))))
    # 获取当前MRI volume的空间尺寸
    # volume形状：[slice,height,width]
    height, width = volume.shape[-2:]

    # 计算高度方向需要补多少像素 # 如果原图高度已经大于目标高度，则不需要补
    pad_height = max(0, target_height - height)
    # 计算宽度方向需要补多少像素 # 如果原图宽度已经大于目标宽度，则不需要补
    pad_width = max(0, target_width - width)

    # 如果原始视野(FOV)小于目标裁剪区域，则先进行padding
    if pad_height or pad_width:

        # 原图不足140mm时进行左右/上下对称补零
        # 避免直接放大较小视野导致空间尺度错误
        volume = np.pad(
            volume,

            # 三个维度分别对应：
            # slice维度不补，height方向补，width方向补
            (
                (0, 0),
                # height方向上下对称补零
                (pad_height // 2, pad_height - pad_height // 2),
                # width方向左右对称补零
                (pad_width // 2, pad_width - pad_width // 2),
            ),

            # 使用0填充，代表MRI背景区域
            mode="constant",
            # 指定填充值为0
            constant_values=0,
        )
        # padding之后重新获取新的height和width
        height, width = volume.shape[-2:]

    # 计算中心裁剪开始位置的高度坐标
    top = (height - target_height) // 2

    # 计算中心裁剪开始位置的宽度坐标
    left = (width - target_width) // 2

    # 对所有slice使用同一个中心区域进行裁剪
    # 保证整个3D volume的空间位置保持一致
    return volume[:, top:top + target_height, left:left + target_width]


def _contrast_features(te, tr, fat_suppression):
    """依据 EDA 使用的 TE/TR 启发式生成 T1、PD、T2、未知四维标记。"""
    # 这些是经验规则推断的扫描对比度类别，不是人工标注的真值。
    if fat_suppression:
        return [0, 1, 1, 0]
    if te <= 0 or tr <= 0:
        return [0, 0, 0, 1]
    if tr < 1000 and te < 30:
        return [1, 0, 0, 0]
    if te < 60:
        return [0, 1, 0, 0]
    return [0, 0, 1, 0]


def _binary_flag(value):
    value = pd.to_numeric(value, errors="coerce")
    return int(value) if pd.notna(value) and value in (0, 1) else None


def read_series_quality(series_path):
    """只读 header 估计几何质量；不解码像素，不把 header 完整性当像素损坏率。"""
    files = sorted(Path(series_path).glob("*.dcm"))
    quality = dict.fromkeys(QUALITY_COLUMNS, None)
    quality.update(quality_usable=0, quality_header_errors=0, quality_geometry_errors=0,
                   quality_unique_slices=0)
    headers, normals, locations = [], [], []
    for file in files:
        try:
            header = inference_read_header(file, stop_before_pixels=True)
        except (OSError, ValueError, EOFError, InvalidDicomError):
            quality["quality_header_errors"] += 1
            continue
        try:
            orientation = np.asarray(header.ImageOrientationPatient, dtype=np.float64)
            location = np.asarray(header.ImagePositionPatient, dtype=np.float64)
            if orientation.shape != (6,) or location.shape != (3,):
                raise ValueError("Invalid geometry dimensions")
            if not np.isfinite(orientation).all() or not np.isfinite(location).all():
                raise ValueError("Nonfinite geometry")
            normal = np.cross(orientation[:3], orientation[3:])
            length = float(np.linalg.norm(normal))
            if length < 1e-6:
                raise ValueError("Degenerate orientation")
            normals.append(normal / length)
            locations.append(location)
            headers.append(header)
        except (AttributeError, TypeError, ValueError):
            quality["quality_geometry_errors"] += 1
    if not headers:
        return quality
    normal = normals[0]
    positions = np.asarray(locations) @ normal
    # 合并相同位置的重建/重复切片，不以文件数量奖励重复图像。
    unique = np.unique(np.round(positions, 4))
    quality["quality_unique_slices"] = len(unique)
    quality["quality_coverage_mm"] = float(unique[-1] - unique[0])
    if len(unique) > 1:
        quality["quality_slice_spacing_mm"] = float(np.median(np.diff(unique)))
    ordered_headers = [headers[i] for i in np.argsort(positions)]
    try:
        spacing = _get_pixel_spacing(ordered_headers, series_path)
        quality["quality_pixel_spacing_mm"] = float(spacing.max())
        middle = ordered_headers[len(ordered_headers) // 2]
        rows, columns = int(middle.Rows), int(middle.Columns)
        if rows <= 0 or columns <= 0:
            raise ValueError("Invalid image matrix")
        quality["quality_fov_mm"] = float(min(rows * spacing[0], columns * spacing[1]))
    except (AttributeError, TypeError, ValueError):
        quality["quality_geometry_errors"] += 1
    aligned = bool(np.all(np.abs(np.asarray(normals) @ normal) >= 0.999))
    quality["quality_usable"] = int(
        len(unique) >= 2 and aligned and quality["quality_header_errors"] == 0 and
        quality["quality_geometry_errors"] == 0 and quality["quality_fov_mm"] is not None
    )
    return quality


def _cached_series_quality(series_path, cache_dir):
    directory = Path(series_path)
    cache_path = None
    if cache_dir is not None:
        files = []
        for file in sorted(directory.glob("*.dcm")):
            stat = file.stat()
            files.append((file.name, stat.st_size, stat.st_mtime_ns))
        signature = json.dumps([SERIES_SELECTION_VERSION, str(directory.resolve()), files])
        cache_path = Path(cache_dir) / (hashlib.sha256(signature.encode()).hexdigest() + ".json")
        if cache_path.is_file():
            try:
                cached = json.loads(cache_path.read_text(encoding="utf-8"))
                if (isinstance(cached, dict) and cached.get("version") == SERIES_SELECTION_VERSION and
                    isinstance(cached.get("quality"), dict) and all(k in cached["quality"] for k in QUALITY_COLUMNS)):
                    return cached["quality"]
            except (OSError, ValueError, KeyError, TypeError):
                pass
    quality = read_series_quality(directory)
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        temporary = cache_path.with_name(f"{cache_path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp")
        try:
            temporary.write_text(json.dumps(dict(version=SERIES_SELECTION_VERSION, quality=quality), allow_nan=False),
                                 encoding="utf-8")
            os.replace(temporary, cache_path)
        finally:
            temporary.unlink(missing_ok=True)
    return quality


def add_series_quality(series_df, cache_dir=None, workers=8):
    """训练由 rank 0 扫描/读取缓存后广播；推理只扫描当前 GPU 分配的序列。"""
    if workers < 1:
        raise ValueError("Quality workers must be positive")
    rows = series_df.copy()
    print(f"Series quality: indexing {len(rows)} series (header-only, workers={workers})", flush=True)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        results = []
        for index, quality in enumerate(pool.map(lambda p: _cached_series_quality(p, cache_dir), rows.SeriesPath), 1):
            results.append(quality)
            if index % 1000 == 0 or index == len(rows):
                print(f"Series quality: {index}/{len(rows)}", flush=True)
    for column in QUALITY_COLUMNS:
        rows[column] = [quality[column] for quality in results]
    return rows


def make_series_selection_config(train_series_df, coverage_quantile=0.2):
    """只使用弱标签训练 Study 的有效几何统计，确定各平面的覆盖参考阈值。"""
    if not 0 <= coverage_quantile <= 1:
        raise ValueError("Coverage quantile must be in [0,1]")
    thresholds = {}
    for plane in ("Sagittal", "Coronal", "Axial"):
        values = train_series_df.loc[
            train_series_df.Anatomical_Plane.eq(plane) & train_series_df.quality_usable.eq(1),
            "quality_coverage_mm",
        ]
        values = pd.to_numeric(values, errors="coerce")
        values = values[np.isfinite(values) & (values > 0)]
        thresholds[plane] = float(values.quantile(coverage_quantile)) if len(values) else 0.0
    config = dict(version=SERIES_SELECTION_VERSION, coverage_quantile=float(coverage_quantile),
                  coverage_thresholds=thresholds)
    validate_series_selection_config(config)
    return config


def validate_series_selection_config(config):
    if not isinstance(config, dict) or config.get("version") != SERIES_SELECTION_VERSION:
        raise ValueError("v14 requires its checkpoint's header-quality series selection config")
    quantile = config.get("coverage_quantile")
    if not isinstance(quantile, (int, float)) or not np.isfinite(quantile) or not 0 <= quantile <= 1:
        raise ValueError("Invalid coverage quantile")
    thresholds = config.get("coverage_thresholds", {})
    if not isinstance(thresholds, dict):
        raise ValueError("Invalid coverage thresholds")
    for plane in ("Sagittal", "Coronal", "Axial"):
        value = thresholds.get(plane)
        if not isinstance(value, (int, float)) or not np.isfinite(value) or value < 0:
            raise ValueError(f"Invalid coverage threshold for {plane}")


def _series_quality_rank(record, config, crop_mm, image_size):
    threshold = config["coverage_thresholds"][record.Anatomical_Plane]
    coverage = float(record.quality_coverage_mm)
    # 覆盖/FOV 达标后不再奖励更大范围，避免优先选大视野而低分辨率的序列。
    coverage_score = min(coverage / threshold, 1.0) if threshold > 0 else 1.0
    fov_score = min(float(record.quality_fov_mm) / crop_mm, 1.0)
    effective_spacing = max(float(record.quality_pixel_spacing_mm), crop_mm / image_size)
    return (coverage_score, fov_score, -effective_spacing,
            -float(record.quality_slice_spacing_mm), int(record.quality_unique_slices))


def select_slot_series(rows, series_selection=None, crop_mm=DEFAULT_CROP_MM, image_size=DEFAULT_IMAGE_SIZE):
    """v10 候选范围不变；配置存在时仅在同一集合内按 header 几何质量排序。"""
    if series_selection is not None:
        validate_series_selection_config(series_selection)
        if any(column not in rows for column in QUALITY_COLUMNS):
            raise ValueError("Call add_series_quality before v14 series selection")
    used, selected = set(), []
    for plane, preference, _ in SLOT_SPECS:
        candidates = rows[(rows.Anatomical_Plane == plane) & (rows.NumSlices > 0)]
        candidates = candidates[~candidates.SeriesInstanceUID.astype(str).isin(used)]
        if preference is not None:
            preferred = candidates[candidates.Fluid_Sensitive.map(_binary_flag).eq(preference)]
            if len(preferred):
                candidates = preferred
        record = candidates.iloc[0] if len(candidates) else None
        if series_selection is not None and len(candidates) > 1:
            usable = candidates[candidates.quality_usable.eq(1)]
            if len(usable):
                # 同分保留首个元素；全部不可用时也回退 v10 CSV 顺序。
                record = max((row for _, row in usable.iterrows()),
                             key=lambda row: _series_quality_rank(row, series_selection, crop_mm, image_size))
        selected.append(record)
        if record is not None:
            used.add(str(record.SeriesInstanceUID))
    return selected


def series_selection_report(series_df, series_selection, crop_mm=DEFAULT_CROP_MM, image_size=DEFAULT_IMAGE_SIZE):
    records = []
    for uid, rows in series_df.groupby("StudyInstanceUID", sort=False):
        original = select_slot_series(rows)
        selected = select_slot_series(rows, series_selection, crop_mm, image_size)
        for slot, old, new in zip(SLOTS, original, selected):
            old_uid = str(old.SeriesInstanceUID) if old is not None else ""
            new_uid = str(new.SeriesInstanceUID) if new is not None else ""
            record = dict(StudyInstanceUID=str(uid), slot=slot, original_series_uid=old_uid,
                          selected_series_uid=new_uid, changed=int(old_uid != new_uid))
            for prefix, series in (("original", old), ("selected", new)):
                for column in QUALITY_COLUMNS:
                    record[f"{prefix}_{column}"] = series[column] if series is not None else None
            records.append(record)
    return pd.DataFrame(records)


def load_series(series_path, num_windows, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,
                fluid_sensitive=0, fat_suppression=0, span_lo=0.02, span_hi=0.98):
    """返回唯一切片缓存、窗口索引、原始位置及元数据；不跨序列构造窗口。"""
    if num_windows < 1 or not 0 <= span_lo < span_hi <= 1:
        raise ValueError("Invalid candidate count or span")
    files, positions, headers = get_sorted_dicom_info(series_path)
    first = int(len(files) * span_lo)
    last = max(first, int(len(files) * span_hi) - 1)
    # 覆盖 2%-98% 的原始索引范围；范围内继续采用 v1 物理位置采样。
    anchors = first + _sample_by_position(positions[first:last + 1], num_windows)
    neighbors = np.clip(anchors[:, None] + np.array([-1, 0, 1]), 0, len(files) - 1)
    needed = np.unique(neighbors)
    spacing = _get_pixel_spacing(headers, series_path)
    volume = physical_center_crop(_read_sampled_slices(files, needed), spacing, crop_mm)
    anchor_volume = volume[np.searchsorted(needed, anchors)]
    foreground = anchor_volume[np.isfinite(anchor_volume) & (anchor_volume != 0)]
    if not foreground.size:
        raise ValueError(f"Series contains no finite non-zero anchor pixels after crop: {series_path}")
    low, high = np.percentile(foreground, [0.5, 99.5])
    volume = np.clip((volume - low) / (high - low + 1e-6), 0, 1).astype(np.float32)
    slices = F.interpolate(torch.from_numpy(volume)[:, None], size=(image_size, image_size),
                           mode="bilinear", align_corners=False)[:, 0]
    window_indices = torch.from_numpy(np.searchsorted(needed, neighbors))
    coverage = float(positions[-1] - positions[0])
    anchor_positions = (2 * (positions[anchors] - positions[0]) / coverage - 1
                        if coverage > 0 else np.zeros(num_windows))
    header = headers[len(headers) // 2]
    te, tr = float(getattr(header, "EchoTime", 0) or 0), float(getattr(header, "RepetitionTime", 0) or 0)
    slice_spacing = float(np.median(np.abs(np.diff(positions)))) if len(positions) > 1 else 0
    features = torch.tensor([
        float(fluid_sensitive), float(fat_suppression), *_contrast_features(te, tr, int(fat_suppression)),
        te / 100, tr / 5000, float(spacing.mean()), slice_spacing / 10, coverage / 200,
    ], dtype=torch.float32)
    return slices, window_indices, torch.as_tensor(anchor_positions, dtype=torch.float32), features


def sample_training_windows(slot_indices, count, rng):
    """每个有效 slot 至少一个；剩余从未选候选随机抽取，候选不足时才重复。"""
    slot_indices = np.asarray(slot_indices)
    slots = np.unique(slot_indices)
    if not len(slots) or count < len(slots):
        raise ValueError("Training window count must cover every valid slot")
    mandatory = np.array([rng.choice(np.flatnonzero(slot_indices == slot)) for slot in slots], dtype=np.int64)
    remaining = np.setdiff1d(np.arange(len(slot_indices)), mandatory)
    rng.shuffle(remaining)
    take = min(count - len(mandatory), len(remaining))
    chosen = np.concatenate([mandatory, remaining[:take]])
    if len(chosen) < count:
        chosen = np.concatenate([chosen, rng.choice(len(slot_indices), count - len(chosen), replace=True)])
    return chosen


class KneeDataset(Dataset):
    """v14 质量排序选序列；训练随机抽24，验证/推理取全部候选。"""
    def __init__(self, study_df, series_df, image_size=DEFAULT_IMAGE_SIZE, crop_mm=DEFAULT_CROP_MM,
                 train=False, train_windows=24, span_lo=0.02, span_hi=0.98, cache_dir=None, seed=42,
                 series_selection=None):
        if train_windows < len(SLOTS) or image_size < 1 or not 0 <= span_lo < span_hi <= 1:
            raise ValueError("train_windows must be >=5; image_size/span must be valid")
        self.study_df = study_df.reset_index(drop=True)
        self.series_groups = {str(uid): group.reset_index(drop=True)
                              for uid, group in series_df.groupby("StudyInstanceUID")}
        self.image_size, self.crop_mm = image_size, crop_mm
        self.train, self.train_windows, self.seed = train, train_windows, seed
        validate_series_selection_config(series_selection)
        self.series_selection = dict(series_selection, coverage_thresholds=dict(series_selection["coverage_thresholds"]))
        self.span_lo, self.span_hi = span_lo, span_hi
        self.cache_dir = Path(cache_dir) if cache_dir is not None else None
        # persistent_workers 的 Dataset 副本共享 epoch，抽样不依赖 worker 数或访问顺序。
        self.epoch = torch.zeros((), dtype=torch.int64).share_memory_()

    def __len__(self):
        return len(self.study_df)

    def set_epoch(self, epoch):
        self.epoch.fill_(epoch)

    def _cache_path(self, uid, selected):
        if self.cache_dir is None:
            return None
        sources = []
        for record in selected:
            if record is None:
                sources.append(None)
                continue
            directory = Path(record.SeriesPath).resolve()
            files = []
            for file in sorted(directory.glob("*.dcm")):
                stat = file.stat()
                files.append((file.name, stat.st_size, stat.st_mtime_ns))
            sources.append((str(directory), _binary_flag(record.Fluid_Sensitive),
                            _binary_flag(record.Fat_Suppression), files))
        signature = json.dumps([CACHE_SCHEMA, str(uid), self.image_size, self.crop_mm,
                                self.span_lo, self.span_hi, SLOT_SPECS, sources], ensure_ascii=False)
        return self.cache_dir / (hashlib.sha256(signature.encode("utf-8")).hexdigest() + ".pt")

    def _build_bank(self, selected):
        pixels, indices, positions, slots = [], [], [], []
        metadata = torch.zeros(len(SLOTS), len(SERIES_FEATURES))
        slot_mask = torch.zeros(len(SLOTS))
        offset = 0
        for slot, (record, (_, _, budget)) in enumerate(zip(selected, SLOT_SPECS)):
            if record is None:
                continue
            slices, windows, depth, features = load_series(
                record.SeriesPath, budget, self.image_size, self.crop_mm,
                _binary_flag(record.Fluid_Sensitive) or 0, _binary_flag(record.Fat_Suppression) or 0,
                self.span_lo, self.span_hi,
            )
            pixels.append(slices)
            indices.append(windows + offset)
            positions.append(depth)
            slots.append(torch.full((budget,), slot, dtype=torch.long))
            metadata[slot], slot_mask[slot] = features, 1
            offset += len(slices)
        if not pixels:
            raise ValueError("Study has no usable series in the five slots")
        return dict(cache_schema=CACHE_SCHEMA, slice_images=torch.cat(pixels),
                    window_indices=torch.cat(indices), window_positions=torch.cat(positions),
                    window_slot_indices=torch.cat(slots), series_features=metadata, slot_mask=slot_mask)

    def candidate_bank(self, uid):
        selected = select_slot_series(self.series_groups[str(uid)], self.series_selection, self.crop_mm, self.image_size)
        path = self._cache_path(uid, selected)
        if path is not None and path.is_file():
            bank = torch.load(path, map_location="cpu", weights_only=True, mmap=True)
            if bank.get("cache_schema") != CACHE_SCHEMA:
                raise ValueError(f"Invalid input cache: {path}")
            return bank
        bank = self._build_bank(selected)
        if path is not None:
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_name(f"{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp")
            try:
                torch.save(bank, temporary)
                try:
                    os.replace(temporary, path)
                except OSError:
                    # 另一 DDP worker 已经写入并 mmap 时，Windows 可能拒绝覆盖。
                    if not path.is_file():
                        raise
            finally:
                temporary.unlink(missing_ok=True)
        return bank

    def __getitem__(self, index):
        row = self.study_df.iloc[index]
        uid = row.StudyInstanceUID
        bank = self.candidate_bank(uid)
        count = len(bank["window_indices"])
        slots = bank["window_slot_indices"].numpy()
        if self.train:
            digest = hashlib.blake2b(f"{self.seed}|{int(self.epoch)}|{uid}".encode(), digest_size=8).digest()
            rng = np.random.default_rng(int.from_bytes(digest, "little"))
            chosen = sample_training_windows(slots, self.train_windows, rng)
        else:
            chosen = np.arange(count)
        # 序列内按原始物理位置排序；不把随机抽取次序当作空间次序。
        positions = bank["window_positions"].numpy()
        chosen = chosen[np.lexsort((chosen, positions[chosen], slots[chosen]))]
        chosen = torch.as_tensor(chosen, dtype=torch.long)
        values = (pd.to_numeric(row[LABELS], errors="coerce").to_numpy(dtype=np.float32)
                  if set(LABELS).issubset(row.index) else np.full(len(LABELS), np.nan, dtype=np.float32))
        valid = ~np.isnan(values)
        return dict(study_uid=str(uid), images=bank["slice_images"][bank["window_indices"][chosen]],
                    window_positions=bank["window_positions"][chosen],
                    window_slot_indices=bank["window_slot_indices"][chosen],
                    slot_mask=bank["slot_mask"], series_features=bank["series_features"],
                    candidate_count=torch.tensor(count),
                    targets=torch.from_numpy(np.nan_to_num(values, nan=0)),
                    label_mask=torch.from_numpy(valid.astype(np.float32)),
                    label_weight=torch.from_numpy(np.where(valid, 2 * np.abs(values - 0.5), 0).astype(np.float32)))


def collate_studies(samples):
    """只拼有效窗口；不同 Study 验证候选数不同时也不补齐图像。"""
    batch = {key: torch.cat([sample[key] for sample in samples])
             for key in ("images", "window_positions", "window_slot_indices")}
    batch["window_batch_indices"] = torch.cat([
        torch.full((len(sample["images"]),), i, dtype=torch.long) for i, sample in enumerate(samples)
    ])
    for key in ("slot_mask", "series_features", "targets", "label_mask", "label_weight", "candidate_count"):
        batch[key] = torch.stack([sample[key] for sample in samples])
    batch["study_uid"] = [sample["study_uid"] for sample in samples]
    return batch


def build_datasets(root=DEFAULT_ROOT, **kwargs):
    metadata = load_metadata(root)
    train_df, train_series_df, test_df, test_series_df, sample_submission = metadata
    quality_cache = Path(kwargs["cache_dir"]) / "series_quality" if kwargs.get("cache_dir") is not None else None
    train_series_df = add_series_quality(train_series_df, quality_cache)
    test_series_df = add_series_quality(test_series_df, quality_cache)
    if kwargs.get("series_selection") is None:
        weak_uids = train_df.loc[~train_df[LABELS].notna().all(axis=1), "StudyInstanceUID"]
        kwargs["series_selection"] = make_series_selection_config(train_series_df[train_series_df.StudyInstanceUID.isin(weak_uids)])
    metadata = (train_df, train_series_df, test_df, test_series_df, sample_submission)
    return (KneeDataset(train_df, train_series_df, train=True, **kwargs),
            KneeDataset(test_df, test_series_df, train=False, **kwargs), sample_submission, metadata)


__all__ = ["DEFAULT_ROOT", "DEFAULT_IMAGE_SIZE", "DEFAULT_CROP_MM", "LABELS", "SLOTS", "SLOT_SPECS",
           "CANDIDATE_BUDGETS", "load_metadata", "get_sorted_dicom_info", "read_dicom", "physical_center_crop",
           "load_series", "select_slot_series", "sample_training_windows", "KneeDataset", "collate_studies",
           "SERIES_SELECTION_VERSION", "QUALITY_COLUMNS", "read_series_quality", "add_series_quality",
           "make_series_selection_config", "validate_series_selection_config", "series_selection_report"]


In [ ]:
"""v1 骨干及层次 Transformer，五slot原始相邻候选与逐标签聚合。"""

from contextlib import nullcontext
from pathlib import Path

import torch
import torch.nn as nn
from transformers import AutoModel


ARCHITECTURE = "baseline_v14_80_candidates_quality_selection_ema"


# 整个模型的数据流：Study 内各序列的切片 → DINOv2 → 切片聚合 → 序列融合 → 12 个标签的 logit。
# 这里的“切片注意力”“序列 Transformer”“标签交叉注意力”是三个不同层级的操作。
def load_backbone(model_dir):
    """Load the local Kaggle/Hugging Face DINOv2 model without network access."""
    model_dir = Path(model_dir)
    # 先明确检查本地权重是否齐全，避免 from_pretrained 隐式尝试联网。
    required = [model_dir / "config.json", model_dir / "pytorch_model.bin"]
    missing = [path.name for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"DINOv2 model directory {model_dir} is missing {missing}")
    return AutoModel.from_pretrained(str(model_dir), local_files_only=True)


class RSNADINOv2(nn.Module):
    """原始相邻窗口 + 序列内物理位置 + 逐标签窗口/slot 聚合。"""

    def __init__(
        self,
        model_dir=None,
        hidden_dim=256,
        num_heads=8,
        slice_layers=2,
        slot_layers=1,
        dropout=0.1,
        freeze_backbone=True,
        encoder_chunk_size=24,
    ):
        super().__init__()
        # DINOv2 对每张 2.5D 图像独立编码；它不直接处理整段 MRI 序列。
        self.backbone = load_backbone(model_dir)

        if encoder_chunk_size < 1:
            raise ValueError("encoder_chunk_size must be positive")
        # 一次编码的图像张数，控制显存占用，不改变模型结构。
        self.encoder_chunk_size = encoder_chunk_size
        self.freeze_backbone(freeze_backbone)
        backbone_dim = self.backbone.config.hidden_size

        # 2.5D 三通道输入沿用 ImageNet 均值/标准差；buffer 会随模型移动设备，但不训练、不存 checkpoint。
        self.register_buffer("image_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("image_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        # CLS 特征由 DINOv2 维度投影到后续 Transformer 使用的 hidden_dim。
        self.slice_projection = nn.Sequential(nn.LayerNorm(backbone_dim), nn.Linear(backbone_dim, hidden_dim))
        # 随机窗口数/位置可变，编码原始 anchor 物理深度，不使用随机抽取次序。
        self.window_position = nn.Sequential(nn.Linear(1, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, hidden_dim))

        # 切片级自注意力：同一序列内的切片可以交换信息，输入形状为 [B×槽位数, S, hidden_dim]。
        slice_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slice_encoder = nn.TransformerEncoder(slice_layer, slice_layers, enable_nested_tensor=False)
        # 每个标签独立选择窗口；常数 bias 会被窗口 softmax 抵消。
        self.slice_attention = nn.Sequential(nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, len(LABELS), bias=False))

        # 将 TE/TR、像素间距等 11 维序列元数据映射到和图像特征相同的维度。
        self.metadata_projection = nn.Sequential(
            nn.Linear(len(SERIES_FEATURES), hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim),
        )
        # 五个 slot 各有身份向量，区分方位与 Fluid Sensitive 选择偏好。
        self.slot_embedding = nn.Parameter(torch.zeros(1, len(SLOTS), hidden_dim))
        # 序列级自注意力：让同一 Study 中有效的不同序列相互交换信息。
        slot_layer = nn.TransformerEncoderLayer(hidden_dim, num_heads, hidden_dim * 4, dropout, batch_first=True, norm_first=True)
        self.slot_encoder = nn.TransformerEncoder(slot_layer, slot_layers, enable_nested_tensor=False)

        # 每个异常标签都有独立 query，用交叉注意力从各序列读取与该标签有关的信息。
        self.label_queries = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_attention = nn.MultiheadAttention(hidden_dim, num_heads, dropout=dropout, batch_first=True)
        # 标签特征归一化后，与逐标签权重做点积，得到未经过 sigmoid 的 12 个 logit。
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.label_weight = nn.Parameter(torch.empty(len(LABELS), hidden_dim))
        self.label_bias = nn.Parameter(torch.zeros(len(LABELS)))
        nn.init.trunc_normal_(self.slot_embedding, std=0.02)
        nn.init.trunc_normal_(self.label_queries, std=0.02)
        nn.init.trunc_normal_(self.label_weight, std=0.02)

    def freeze_backbone(self, freeze=True):
        # 只改变 DINOv2 参数的 requires_grad；后续聚合与预测头始终可训练。
        for parameter in self.backbone.parameters():
            parameter.requires_grad = not freeze

    def unfreeze_last_blocks(self, num_blocks=2):
        # 先冻结整个 backbone，再开放最后 num_blocks 个编码层和最终 LayerNorm。
        self.freeze_backbone(True)
        for block in self.backbone.encoder.layer[-num_blocks:]:
            for parameter in block.parameters():
                parameter.requires_grad = True
        for parameter in self.backbone.layernorm.parameters():
            parameter.requires_grad = True

    def encode_images(self, images):
        use_grad = any(parameter.requires_grad for parameter in self.backbone.parameters())
        with nullcontext() if use_grad else torch.no_grad():
            chunks = [self.backbone(pixel_values=(chunk - self.image_mean) / self.image_std).last_hidden_state[:, 0]
                      for chunk in images.split(self.encoder_chunk_size)]
        return torch.cat(chunks)

    def aggregate_windows(self, features, positions, batch_indices, slot_indices, batch_size):
        """仅有效 slot 进入 slice Transformer，特征补齐处同时屏蔽 Transformer 和池化。"""
        # 输入 features: [N,256]；其余三个索引/位置张量均为 [N]，N 是整个 batch 的有效窗口总数。
        # 用 Study 编号和 slot 编号确定分组；不同 Study 或不同 slot 的窗口不在这里交换信息。
        group_ids = batch_indices * len(SLOTS) + slot_indices
        # 即使调用方重排窗口，也恢复序列内空间顺序。
        order = positions.argsort(stable=True)
        # 稳定排序先按物理位置、再按分组，使同组窗口连续排列并保留组内空间顺序。
        order = order[group_ids[order].argsort(stable=True)]
        group_ids = group_ids[order]
        # 原始 anchor 的归一化物理深度 [N,1] 经 MLP 变为 [N,256]，与图像特征相加。
        positioned = features + self.window_position(positions.to(features.dtype).unsqueeze(-1))
        positioned = positioned[order]
        # 每个 Study-slot 的窗口数可能不同；只让至少有一个窗口的 R 个分组进入 Transformer。
        counts = torch.bincount(group_ids, minlength=batch_size * len(SLOTS))
        active = counts.nonzero(as_tuple=False).squeeze(1)
        active_counts = counts[active]
        # W 是当前 batch 中单个 slot 的最大窗口数，仅把特征补齐到 W，不补齐或编码图像。
        width = int(active_counts.max())
        # 根据每组起点，将连续的窗口特征映射到二维的“分组行号、组内列号”。
        starts = active_counts.cumsum(0) - active_counts
        row_indices = torch.repeat_interleave(torch.arange(len(active), device=features.device), active_counts)
        local_indices = torch.arange(len(features), device=features.device) - torch.repeat_interleave(starts, active_counts)
        flat_indices = row_indices * width + local_indices
        # 补齐后的特征为 [R,W,256]；mask: [R,W]，True 表示真实窗口。
        padded = features.new_zeros(len(active) * width, features.shape[-1])
        padded = padded.index_copy(0, flat_indices, positioned).view(len(active), width, -1)
        mask = torch.arange(width, device=features.device)[None, :] < active_counts[:, None]
        # 两层 slice Transformer 在各 slot 内建模窗口关系；padding 不参与注意力读取。
        contextual = self.slice_encoder(padded, src_key_padding_mask=~mask)
        # 每个窗口输出 12 个标签分数 [R,W,12]；padding 设为 -inf，池化权重为零。
        scores = self.slice_attention(contextual).masked_fill(~mask[..., None], -torch.inf)
        # 每个标签独立在窗口维 W 上做 softmax，再加权求和，得到 [R,12,256]。
        # 同一 slot 因而有 12 个向量，而不是所有标签共用一个序列向量。
        pooled = torch.einsum("rwl,rwd->rld", scores.softmax(dim=1), contextual)
        # 将有效分组放回完整结构 [B,5,12,256]；缺失 slot 保留零特征并返回有效 mask [B,5]。
        slots = features.new_zeros(batch_size * len(SLOTS), len(LABELS), features.shape[-1])
        slots = slots.index_copy(0, active, pooled).view(batch_size, len(SLOTS), len(LABELS), -1)
        return slots, counts.view(batch_size, len(SLOTS)) > 0

    def forward(self, images, slot_mask, series_features=None, *, window_positions,
                window_batch_indices, window_slot_indices):
        # images: [N,3,H,W]，三通道是原始相邻切片；slot_mask: [B,5]；元数据: [B,5,11]。
        # 窗口以紧凑形式拼接，三个 [N] 张量分别记录物理位置、所属 Study 和所属 slot。
        if images.ndim != 4 or images.shape[1] != 3 or not len(images):
            raise ValueError("Expected nonempty packed windows [N,3,H,W]")
        if slot_mask.ndim != 2 or slot_mask.shape[1] != len(SLOTS):
            raise ValueError("Expected slot_mask [B,5]")
        batch_size = len(slot_mask)
        # 每个 Study 至少有一个有效 slot，且每张图像都必须有完整、合法的窗口映射。
        if not bool(slot_mask.bool().any(dim=1).all()):
            raise ValueError("Every study must have at least one valid slot")
        for values in (window_positions, window_batch_indices, window_slot_indices):
            if values.shape != (len(images),):
                raise ValueError("Window maps must match the packed images")
        if (bool((window_batch_indices < 0).any()) or bool((window_batch_indices >= batch_size).any()) or
            bool((window_slot_indices < 0).any()) or bool((window_slot_indices >= len(SLOTS)).any())):
            raise ValueError("Invalid window batch/slot index")
        # DINOv2 独立编码每个窗口的 CLS，再通过 LayerNorm + Linear 投影为 [N,256]。
        features = self.slice_projection(self.encode_images(images))
        # 序列内 Transformer + 逐标签窗口池化，得到 slot_features: [B,5,12,256]。
        slot_features, present = self.aggregate_windows(
            features, window_positions, window_batch_indices, window_slot_indices, batch_size
        )
        # 核对实际窗口分组与数据提供的 slot_mask，防止有效 slot 没有窗口或缺失 slot 混入窗口。
        if not torch.equal(present, slot_mask.bool()):
            raise ValueError("Every valid slot must have a window; invalid slots cannot contain windows")
        if series_features is not None:
            # 11 维元数据投影为 [B,5,256]，扩展标签维后加到该 slot 的全部 12 个标签特征上。
            slot_features = slot_features + self.metadata_projection(series_features).unsqueeze(2)
        # slot 身份向量区分五种序列角色，同一个 slot 的身份信息由各标签共享。
        slot_features = slot_features + self.slot_embedding.unsqueeze(2)
        # [B,5,12,256] → [B,12,5,256] → [B×12,5,256]，每个 Study-label 独立融合五个 slot。
        label_slots = slot_features.permute(0, 2, 1, 3).reshape(batch_size * len(LABELS), len(SLOTS), -1)
        # 缺失 slot 的 mask 扩展到所有标签，True 表示 Transformer/交叉注意力应忽略该 slot。
        label_mask = (~present)[:, None].expand(batch_size, len(LABELS), len(SLOTS))
        label_mask = label_mask.reshape(batch_size * len(LABELS), len(SLOTS))
        # 一层 slot Transformer：各标签的五个 slot 分别交换信息，但共用同一套网络参数。
        label_slots = self.slot_encoder(label_slots, src_key_padding_mask=label_mask)
        # 12 个 query 是可学习参数；每个标签取自己的 query，形成 [B×12,1,256]。
        queries = self.label_queries[None, :, None].expand(batch_size, -1, -1, -1)
        queries = queries.reshape(batch_size * len(LABELS), 1, -1)
        # 标签 query 通过交叉注意力读取自己的五个 slot，输出 [B×12,1,256] 标签特征。
        label_features, _ = self.label_attention(queries, label_slots, label_slots,
                                                key_padding_mask=label_mask, need_weights=False)
        # 恢复 [B,12,256] 并归一化，再与逐标签分类权重做点积，返回 logits [B,12]。
        label_features = self.output_norm(label_features.view(batch_size, len(LABELS), -1))
        return (label_features * self.label_weight[None]).sum(-1) + self.label_bias


def predict_batch(model, batch, use_metadata=True):
    return model(batch["images"], batch["slot_mask"], batch["series_features"] if use_metadata else None,
                 window_positions=batch["window_positions"], window_batch_indices=batch["window_batch_indices"],
                 window_slot_indices=batch["window_slot_indices"])


__all__ = ["ARCHITECTURE", "RSNADINOv2", "predict_batch"]


In [ ]:
"""v20_v1 I/O acceleration; per-model numerics and architecture remain v14.

Reference: https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/discussion/743374
"""
import io
import threading
from collections import deque
from copy import copy

DICOM_THREADS = 16
PREFETCH_STUDIES = 2

_study_io = threading.local()
_pydicom_dcmread = pydicom.dcmread


def _read_dicom_record(path):
    # Keep the v14 parser and decoder. Only the disk read is shared.
    try:
        raw = Path(path).read_bytes()
        header = _pydicom_dcmread(io.BytesIO(raw), stop_before_pixels=True)
        return raw, header, None
    except (OSError, ValueError, EOFError, InvalidDicomError) as error:
        return None, None, error


def _record(path):
    records = getattr(_study_io, "records", None)
    return records[Path(path)] if records is not None else _read_dicom_record(path)


def inference_read_header(path, stop_before_pixels=True):
    _, header, error = _record(path)
    if error is not None:
        raise error
    return header


def inference_read_dataset(path):
    raw, _, error = _record(path)
    if error is not None:
        raise error
    return _pydicom_dcmread(io.BytesIO(raw))


class FastInferenceDataset(KneeDataset):
    """One concurrent file read feeds quality ranking, ordering and decoding."""
    def __init__(self, *args, io_pool, **kwargs):
        super().__init__(*args, **kwargs)
        self.io_pool = io_pool

    @torch.inference_mode()
    def __getitem__(self, index):
        uid = str(self.study_df.iloc[index].StudyInstanceUID)
        rows = self.series_groups[uid]
        files = [file for directory in rows.SeriesPath for file in Path(directory).glob('*.dcm')]
        _study_io.records = dict(zip(files, self.io_pool.map(_read_dicom_record, files)))
        try:
            return super().__getitem__(index)
        finally:
            # Retain only the finished CPU sample, never source DICOM bytes.
            del _study_io.records

    def candidate_bank(self, uid):
        quality_cache = getattr(_study_io, "quality_rows", {})
        if uid in quality_cache:
            rows = quality_cache[uid]
        else:
            rows = self.series_groups[str(uid)].copy()
            qualities = [read_series_quality(path) for path in rows.SeriesPath]
            for column in QUALITY_COLUMNS:
                rows[column] = [quality[column] for quality in qualities]
            quality_cache[uid] = rows
        selected = select_slot_series(rows, self.series_selection, self.crop_mm, self.image_size)
        # All folds share preprocessing parameters, but their TRAINING coverage
        # thresholds can select different series. Reuse only identical selections.
        key = tuple(None if row is None else str(row.SeriesInstanceUID) for row in selected)
        banks = getattr(_study_io, "banks", {})
        if key not in banks:
            banks[key] = self._build_bank(selected)
        return banks[key]


class FiveFoldInferenceDataset(FastInferenceDataset):
    def __init__(self, *args, series_selections, **kwargs):
        super().__init__(*args, series_selection=series_selections[0], **kwargs)
        self.series_selections = series_selections

    @torch.inference_mode()
    def __getitem__(self, index):
        uid = str(self.study_df.iloc[index].StudyInstanceUID)
        rows = self.series_groups[uid]
        files = [file for directory in rows.SeriesPath for file in Path(directory).glob("*.dcm")]
        _study_io.records = dict(zip(files, self.io_pool.map(_read_dicom_record, files)))
        _study_io.quality_rows, _study_io.banks = {}, {}
        try:
            samples, sample_cache = [], {}
            for config in self.series_selections:
                view = copy(self)
                view.series_selection = config
                bank = view.candidate_bank(uid)
                key = id(bank)
                if key not in sample_cache:
                    sample_cache[key] = KneeDataset.__getitem__(view, index)
                samples.append(sample_cache[key])
            return samples
        finally:
            del _study_io.records, _study_io.quality_rows, _study_io.banks


def prefetched_batches(dataset, study_pool):
    """Bounded thread prefetch, safe in a notebook; no fork after CUDA init."""
    pending, next_index = deque(), 0
    try:
        while next_index < len(dataset) and len(pending) < PREFETCH_STUDIES:
            pending.append(study_pool.submit(dataset.__getitem__, next_index))
            next_index += 1
        while pending:
            samples = pending.popleft().result()
            batches, batch_cache = [], {}
            for sample in samples:
                key = id(sample)
                if key not in batch_cache:
                    batch = collate_studies([sample])
                    batch_cache[key] = {name: value.pin_memory() if isinstance(value, torch.Tensor) else value
                                        for name, value in batch.items()}
                batches.append(batch_cache[key])
            yield batches
            if next_index < len(dataset):
                pending.append(study_pool.submit(dataset.__getitem__, next_index))
                next_index += 1
    finally:
        for future in pending:
            future.cancel()


CV_METHOD = "v20_v1_weak5_iterative_gold58_selection_v1"
N_FOLDS = 5
GOLD_COUNT = 58

def validate_fold_metadata(metadata):
    """Reject missing/duplicate folds, mixed experiments and holdout leakage."""
    if len(metadata) != N_FOLDS or any(not isinstance(item, dict) for item in metadata):
        raise ValueError("Exactly five v20_v1 cross-validation checkpoints are required")
    if {item.get("fold") for item in metadata} != set(range(N_FOLDS)):
        raise ValueError("Checkpoints must contain each fold 0..4 exactly once")
    first = metadata[0]
    for item in metadata:
        if item.get("method") != CV_METHOD or item.get("n_splits") != N_FOLDS:
            raise ValueError("Checkpoint is not a v20_v1 weak-five-fold/gold-selection model")
        for name in ("split_seed", "manifest_sha256", "data_sha256", "gold_uids"):
            if name not in first or item.get(name) != first[name]:
                raise ValueError(f"Cross-validation metadata mismatch: {name}")
        for name in ("train_uids", "valid_uids", "gold_uids"):
            uids = item.get(name)
            if not isinstance(uids, list) or not uids or any(not isinstance(uid, str) for uid in uids) or len(set(uids)) != len(uids):
                raise ValueError(f"Invalid checkpoint membership: {name}")
        train, valid, gold = (set(item[name]) for name in ("train_uids", "valid_uids", "gold_uids"))
        if len(gold) != GOLD_COUNT or train & valid or gold & (train | valid):
            raise ValueError("Training/validation/gold holdout overlap or wrong gold count")
    all_valid = set().union(*(set(item["valid_uids"]) for item in metadata))
    if len(all_valid) != sum(len(item["valid_uids"]) for item in metadata):
        raise ValueError("Validation studies overlap between folds")
    for item in metadata:
        if set(item["train_uids"]) != all_valid - set(item["valid_uids"]):
            raise ValueError("Each fold must train on exactly the other four weak folds")


In [ ]:
"""v20_v1 five-fold inference: unchanged per-model numerics, mean probabilities."""

import os
import time
from contextlib import ExitStack
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")

import numpy as np
import pandas as pd
import torch

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
# Empty: discover fold0.pt through fold4.pt, or five best.pt files.
# Otherwise specify all five paths, e.g. /kaggle/input/<dataset>/fold0.pt.
CHECKPOINT_PATHS = []
CHECKPOINT_FILENAMES = {"best.pt", *(f"fold{i}.pt" for i in range(N_FOLDS))}
DINOV2_MODEL_DIR = "/kaggle/input/models/metaresearch/dinov2/pytorch/small/1"
OUTPUT_PATH = Path("/kaggle/working/submission.csv")
BATCH_SIZE = 1
ENCODER_CHUNK_SIZE = 16
NUM_GPUS = min(2, torch.cuda.device_count())


def find_one(candidates, description):
    paths = sorted(set(Path(path) for path in candidates))
    if len(paths) != 1:
        raise RuntimeError(f"Expected one {description}, found {len(paths)}: {paths}. Set its path explicitly above.")
    return paths[0]


def input_files():
    for directory, subdirs, files in os.walk("/kaggle/input"):
        subdirs[:] = [name for name in subdirs if name not in {"train_series", "test_series"}]
        for name in files:
            if name in CHECKPOINT_FILENAMES or name in {"test_series.csv", "config.json"}:
                yield Path(directory) / name


def resolve_inputs():
    found = list(input_files()) if not all((DATA_ROOT, CHECKPOINT_PATHS, DINOV2_MODEL_DIR)) else []
    root = Path(DATA_ROOT) if DATA_ROOT else find_one(
        (p.parent for p in found if p.name == "test_series.csv"
         if (p.parent / "test.csv").is_file() and (p.parent / "sample_submission.csv").is_file()
         and (p.parent / "test_series").is_dir()), "competition data directory")
    checkpoints = [Path(path) for path in CHECKPOINT_PATHS] if CHECKPOINT_PATHS else sorted({p for p in found if p.name in CHECKPOINT_FILENAMES})
    if len(checkpoints) != N_FOLDS or len(set(path.resolve() for path in checkpoints)) != N_FOLDS:
        raise ValueError(f"Provide exactly five distinct v20_v1 checkpoints (fold0.pt..fold4.pt or best.pt); found {checkpoints}")
    model_dir = Path(DINOV2_MODEL_DIR) if DINOV2_MODEL_DIR else find_one(
        (p.parent for p in found if p.name == "config.json" and (p.parent / "pytorch_model.bin").is_file()),
        "Hugging Face DINOv2 directory")
    for path in (root / "test.csv", root / "test_series.csv", root / "sample_submission.csv", root / "test_series",
                 *checkpoints, model_dir / "config.json", model_dir / "pytorch_model.bin"):
        if not path.exists():
            raise FileNotFoundError(path)
    return root, checkpoints, model_dir


def load_fold_bundles(paths):
    bundles, common = [], None
    for path in paths:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
        if (checkpoint.get("architecture") != ARCHITECTURE or checkpoint.get("slots") != SLOTS or
                checkpoint.get("candidate_budgets") != CANDIDATE_BUDGETS):
            raise ValueError(f"Model architecture mismatch: {path}")
        config = checkpoint.get("series_selection")
        validate_series_selection_config(config)
        args = checkpoint.get("args")
        if not isinstance(args, dict) or any(name not in args for name in ("image_size", "crop_mm", "span_lo", "span_hi")):
            raise ValueError(f"Missing v14 preprocessing parameters: {path}")
        image_size, crop_mm = int(args["image_size"]), float(args["crop_mm"])
        span_lo, span_hi = float(args["span_lo"]), float(args["span_hi"])
        signature = (image_size, crop_mm, span_lo, span_hi, not bool(args.get("no_metadata", False)))
        if image_size <= 0 or not np.isfinite(crop_mm) or crop_mm <= 0 or not 0 <= span_lo < span_hi <= 1:
            raise ValueError(f"Invalid preprocessing parameters: {path}")
        if common is not None and common != signature:
            raise ValueError("All folds must share image size, crop, span and metadata settings")
        common = signature
        state = checkpoint["model"]
        if any(key.startswith("backbone.patch_embed.") for key in state):
            raise ValueError("Expected the Hugging Face DINOv2 backbone")
        bundles.append(dict(path=str(path), state=state, series_selection=config,
                            cross_validation=checkpoint.get("cross_validation")))
        del checkpoint
    validate_fold_metadata([item["cross_validation"] for item in bundles])
    bundles.sort(key=lambda item: item["cross_validation"]["fold"])
    return bundles, common


def mean_fold_probabilities(scores):
    if len(scores) != N_FOLDS:
        raise ValueError("Exactly five fold probabilities are required")
    stacked = np.stack(scores)
    if not np.isfinite(stacked).all() or not ((stacked >= 0) & (stacked <= 1)).all():
        raise ValueError("Invalid fold probabilities")
    return stacked.mean(axis=0, dtype=np.float64).astype(np.float32)


@torch.inference_mode()
def inference_worker(rank, models, local_test_df, series_df, image_size, crop_mm, root,
                     use_metadata, span_lo, span_hi, series_selections, io_pool, study_pool):
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")
    started = time.monotonic()
    if local_test_df.empty:
        return pd.DataFrame(columns=["StudyInstanceUID", *LABELS])
    if len(models) != N_FOLDS or len(series_selections) != N_FOLDS:
        raise ValueError("Each GPU needs five models and their own series-selection configs")
    rows = series_df[series_df.StudyInstanceUID.isin(local_test_df.StudyInstanceUID)]
    rows = _prepare_series_df(rows, root / "test_series")
    if set(local_test_df.StudyInstanceUID) - set(rows.StudyInstanceUID):
        raise ValueError("Missing series metadata for test studies")
    dataset = FiveFoldInferenceDataset(local_test_df, rows, image_size=image_size, crop_mm=crop_mm,
                        train=False, span_lo=span_lo, span_hi=span_hi, cache_dir=None,
                        series_selections=series_selections, io_pool=io_pool)
    predictions = {}
    for index, fold_batches in enumerate(prefetched_batches(dataset, study_pool), start=1):
        fold_scores, uids = [], None
        for model, batch in zip(models, fold_batches):
            if uids is not None and uids != batch["study_uid"]:
                raise ValueError("Fold batches refer to different studies")
            uids = batch["study_uid"]
            # Exactly the v14 device-copy, autocast, forward and sigmoid steps.
            batch = {key: value.to(device, non_blocking=True) if isinstance(value, torch.Tensor) else value
                     for key, value in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                logits = predict_batch(model, batch, use_metadata=use_metadata)
            scores = torch.sigmoid(logits.float()).cpu().numpy()
            fold_scores.append(scores)
        scores = mean_fold_probabilities(fold_scores)
        for uid, score in zip(uids, scores):
            predictions[str(uid)] = score
        if index == 1 or index % 5 == 0 or index == len(dataset):
            print(f"GPU {rank}: {index}/{len(dataset)} studies (five folds), elapsed {time.monotonic() - started:.1f}s", flush=True)
    if set(predictions) != set(local_test_df.StudyInstanceUID):
        raise ValueError("Incomplete/duplicate study prediction coverage")
    values = np.stack([predictions[uid] for uid in local_test_df.StudyInstanceUID])
    if values.shape != (len(dataset), len(LABELS)):
        raise ValueError("Incorrect prediction shape")
    partial = pd.DataFrame(values, columns=LABELS)
    partial.insert(0, "StudyInstanceUID", local_test_df.StudyInstanceUID.to_numpy())
    return partial


def main():
    if BATCH_SIZE != 1 or DICOM_THREADS < 1 or PREFETCH_STUDIES < 1:
        raise ValueError("Use BATCH_SIZE=1 and positive I/O/prefetch settings")
    root, checkpoint_paths, model_dir = resolve_inputs()
    if NUM_GPUS < 1 or NUM_GPUS > torch.cuda.device_count():
        raise RuntimeError("Enable a Kaggle GPU accelerator")
    test_df = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    series_df = pd.read_csv(root / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    if list(sample.columns) != ["StudyInstanceUID", *LABELS]:
        raise ValueError("Submission columns differ from the training model")
    if test_df.StudyInstanceUID.duplicated().any() or sample.StudyInstanceUID.duplicated().any():
        raise ValueError("Duplicate test/sample study IDs")
    if set(test_df.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("test.csv and sample_submission.csv contain different studies")
    bundles, (image_size, crop_mm, span_lo, span_hi, use_metadata) = load_fold_bundles(checkpoint_paths)
    selections = [item["series_selection"] for item in bundles]
    models = [[] for _ in range(NUM_GPUS)]
    for bundle in bundles:
        fold = bundle["cross_validation"]["fold"]
        print(f"fold={fold}, checkpoint={bundle['path']}, selection={bundle['series_selection']}", flush=True)
        for rank in range(NUM_GPUS):
            model = RSNADINOv2(model_dir=str(model_dir), encoder_chunk_size=ENCODER_CHUNK_SIZE, freeze_backbone=True)
            model.load_state_dict(bundle["state"], strict=True)
            model.to(torch.device(f"cuda:{rank}")).eval()
            models[rank].append(model)
        del bundle["state"]
    del bundles
    with ExitStack() as stack:
        io_pools = [stack.enter_context(ThreadPoolExecutor(max_workers=DICOM_THREADS)) for _ in range(NUM_GPUS)]
        study_pools = [stack.enter_context(ThreadPoolExecutor(max_workers=PREFETCH_STUDIES)) for _ in range(NUM_GPUS)]
        pool = stack.enter_context(ThreadPoolExecutor(max_workers=NUM_GPUS))
        futures = [pool.submit(inference_worker, rank, models[rank],
                               test_df.iloc[rank::NUM_GPUS].reset_index(drop=True), series_df,
                               image_size, crop_mm, root, use_metadata, span_lo, span_hi, selections,
                               io_pools[rank], study_pools[rank]) for rank in range(NUM_GPUS)]
        predictions = pd.concat([future.result() for future in futures], ignore_index=True)
    if predictions.StudyInstanceUID.duplicated().any() or set(predictions.StudyInstanceUID) != set(sample.StudyInstanceUID):
        raise ValueError("GPU predictions do not cover sample_submission exactly once")
    submission = sample[["StudyInstanceUID"]].merge(predictions, on="StudyInstanceUID", how="left", validate="one_to_one")
    if list(submission.columns) != ["StudyInstanceUID", *LABELS] or not np.isfinite(submission[LABELS].to_numpy()).all():
        raise ValueError("Invalid merged submission")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False, lineterminator="\n")
    print(f"Saved {OUTPUT_PATH}: {len(submission)} studies, five-fold mean probabilities")


if __name__ == "__main__":
    main()
